In [ ]:
from pathlib import Path
import pandas as pd

# Find the project and dataset folders
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
DATA_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Columns needed for the research question
columns = [
    "code_module",
    "code_presentation",
    "id_student",
    "date",
    "sum_click",
]

# Process the large file in smaller pieces
chunk_size = 500_000
partial_results = []

for chunk in pd.read_csv(
    DATA_DIR / "studentVle.csv",
    usecols=columns,
    chunksize=chunk_size
):
    # Keep only the first 14 course days
    early = chunk.loc[
        chunk["date"].between(0, 13)
    ]

    # Add clicks for each student in each course presentation
    grouped = (
        early.groupby(
            ["code_module", "code_presentation", "id_student"],
            as_index=False
        )["sum_click"]
        .sum()
    )

    partial_results.append(grouped)

    print(f"Processed {len(chunk):,} activity rows")

# Combine the partial results
early_engagement = pd.concat(
    partial_results,
    ignore_index=True
)

# Sum across chunks to get each student's total clicks
early_engagement = (
    early_engagement.groupby(
        ["code_module", "code_presentation", "id_student"],
        as_index=False
    )["sum_click"]
    .sum()
    .rename(columns={"sum_click": "clicks_first_14_days"})
)

# Save the feature table
early_engagement.to_csv(
    OUTPUT_DIR / "early_engagement.csv",
    index=False
)

print("\nEarly engagement feature table created!")
print("Number of student-course records:", len(early_engagement))
display(early_engagement.head())

In [ ]:
# Load student outcomes
students = pd.read_csv(DATA_DIR / "studentInfo.csv")

# Join on student AND course presentation
research_data = students.merge(
    early_engagement,
    on=["code_module", "code_presentation", "id_student"],
    how="left",
    validate="one_to_one"
)

# Students with no recorded clicks in the window receive zero
research_data["clicks_first_14_days"] = (
    research_data["clicks_first_14_days"].fillna(0)
)

# Define the proposed binary outcome
research_data["at_risk"] = (
    research_data["final_result"]
    .isin(["Fail", "Withdrawn"])
    .astype(int)
)

# Save the merged research table
research_data.to_csv(
    OUTPUT_DIR / "research_dataset.csv",
    index=False
)

print("Research dataset shape:", research_data.shape)
print("\nAt-risk outcome distribution:")
print(
    research_data["at_risk"]
    .value_counts()
    .rename(index={0: "Not at risk", 1: "At risk"})
)

print("\nEarly engagement summary:")
display(
    research_data["clicks_first_14_days"].describe()
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compare early engagement across final outcomes
engagement_summary = (
    research_data.groupby("final_result")["clicks_first_14_days"]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("Early engagement by final outcome:")
display(engagement_summary)

# Compare the distributions visually
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=research_data,
    x="final_result",
    y="clicks_first_14_days",
    order=["Distinction", "Pass", "Fail", "Withdrawn"]
)

plt.title("Early Online Engagement by Final Academic Outcome")
plt.xlabel("Final outcome")
plt.ylabel("Clicks during the first 14 course days")
plt.tight_layout()
plt.show()

In [ ]:
# Compare early engagement for the two proposed risk groups
risk_summary = (
    research_data.groupby("at_risk")["clicks_first_14_days"]
    .agg(["count", "mean", "median"])
    .round(2)
)

risk_summary.index = risk_summary.index.map({
    0: "Not at risk (Pass or Distinction)",
    1: "At risk (Fail or Withdrawn)"
})

print("Early engagement by risk group:")
display(risk_summary)

# Compare the proportion of records with zero recorded clicks
zero_click_summary = (
    research_data.assign(
        zero_clicks=research_data["clicks_first_14_days"].eq(0)
    )
    .groupby("at_risk")["zero_clicks"]
    .mean()
    .mul(100)
    .round(2)
)

zero_click_summary.index = zero_click_summary.index.map({
    0: "Not at risk",
    1: "At risk"
})

print("\nPercentage with zero recorded clicks:")
display(zero_click_summary)